**This updated script incorporates support for analyzinfg GTFS and GTFS-RT dataset formats, after seeing that these two types were heavily used by datasets, especially those with tag "autobus".**

In [194]:
%pip install -q dspy-ai requests==2.32.4 pandas==2.2.2 pypdf scikit-learn gtfs-realtime-bindings protobuf

In [195]:
import os
import dspy
from google.colab import userdata

# ---- USER FILLS THESE IN ----
AZURE_API_KEY = userdata.get('Azure')          # <-- put your key in Colab secrets or env var
AZURE_API_BASE = "https://o3miniapi.cognitiveservices.azure.com/"  # your endpoint
AZURE_API_VERSION = "2024-12-01-preview"
AZURE_DEPLOYMENT = "gpt-5-mini"       # <-- your chat-capable deployment name (e.g., "gpt-4o-mini")

# Recommended: use env var rather than hardcoding.
if AZURE_API_KEY:
    os.environ["AZURE_API_KEY"] = AZURE_API_KEY

# DSPy LM via LiteLLM Azure provider string.
# Note: provider strings can vary across LiteLLM versions; "azure/<deployment>" is the common pattern.
lm = dspy.LM(
    f"azure/{AZURE_DEPLOYMENT}",
    api_key=os.getenv("AZURE_API_KEY", ""),
    api_base=AZURE_API_BASE,
    api_version=AZURE_API_VERSION,
    model_type="chat",
    temperature=1.0,
    max_tokens=32000,
)

dspy.configure(lm=lm)


In [196]:
import requests
import re
from typing import Dict, Any, List, Tuple, Optional, Union

def get_dataset_payload(dataset_id: str) -> Dict[str, Any]:
    url = f"https://www.data.gouv.fr/api/1/datasets/{dataset_id}/"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()


# Now (after having created this new notebook - a copy of the first one) it's
# time to solve for the case when resource url's DO NOT end with the matching
# file extension. I will do this by inspecting whether the url ends with a
# .something or .something/
# in the case that it doesn't, then I will add it to the empty dict "urls_no"
# which just means "urls with no extensions"
# And the reason why I chose "urls_no" to be a dict and not a list is:
# Along with the actual URL, I want to also store the entire "Resources" field
# from the ds.get("resources"), since the function that will be calling this one
# might need the information from that field to decide what type of file this
# download url will contain. If you think Dict is a good choice for this, then
# use Dict, but if its not the best, THEN PICK THE BEST OPTION!

EXT_RE = re.compile(r"/[^/?]+\.[a-zA-Z0-9]+/?$")

def extract_resource_urls(ds: Dict[str, Any]) -> Tuple[List[str], List[Dict[str, Any]]]:
    """
    Returns:
      - urls: URLs that clearly end with a file extension
      - urls_no: structured entries for URLs without extensions
    """
    urls: List[str] = []
    urls_no: List[Dict[str, Any]] = []

    for res in (ds.get("resources") or []):
        u = res.get("url")
        if not u:
            continue

        if EXT_RE.search(u):
            urls.append(u)
        else:
            urls_no.append({
                "url": u,
                "resource": res,
                "reason": "no_file_extension",
            })

    return urls, urls_no


#    for res in (ds.get("community_resources") or []):
#        u = res.get("url")
#        if u:
#            urls.append(u)
    # de-dupe preserving order
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def summarize_dataset_metadata(ds: Dict[str, Any]) -> str:
    org = ds.get("organization") or {}
    lines = [
        f"ID: {ds.get('id','')}",
        f"Title: {ds.get('title','')}",
        f"Description_short: {ds.get('description_short','')}",
        f"Description: {ds.get('description','')}",
        f"Organization: {org.get('name','')} (slug={org.get('slug','')}, id={org.get('id','')})",
        f"License: {ds.get('license','')}",
        f"Frequency: {ds.get('frequency','')}",
        f"Created_at: {ds.get('created_at','')}",
        f"Last_update: {ds.get('last_update','')}",
        f"Tags: {', '.join(ds.get('tags') or [])}",
        f"URI: {ds.get('uri','')}",
        f"Page: {ds.get('page','')}",
    ]
    return "\n".join([x for x in lines if x.strip()])


I will add a chunk of code that could potentially help me with decoding gtfs-rt files (which are usually bin files as well). Look at: https://chatgpt.com/share/69823a9c-6110-8006-a7d3-a07a3da166ee for reference as to how I got the code.

In [197]:
def _meta_text(resource: Optional[Dict[str, Any]]) -> str:
    r = resource or {}
    extras = r.get("extras") or {}
    parts = [
        str(r.get("format") or ""),
        str(r.get("mime") or ""),
        str(r.get("description") or ""),
        str(extras.get("analysis:mime-type") or ""),
    ]
    return " ".join(parts).lower()

def _looks_like_gtfs_rt(resource: Optional[Dict[str, Any]]) -> bool:
    t = _meta_text(resource)
    # You can expand these keywords as you see more real-world cases.
    return any(k in t for k in ["gtfs-rt", "gtfsrt", "gtfs realtime", "realtime", "protobuf", "x-protobuf"])

def extract_gtfs_rt_preview(path: str, max_entities: int = 50) -> str:
    """
    Decode GTFS-RT protobuf into JSON preview.
    Requires: gtfs-realtime-bindings
    """
    try:
        from google.transit import gtfs_realtime_pb2
    except Exception:
        return (
            "[GTFS-RT detected but decoder not installed]\n"
            "Install: pip install gtfs-realtime-bindings protobuf"
        )

    with open(path, "rb") as f:
        data = f.read()

    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(data)

    # Convert to a dict-ish structure manually (keep only useful fields)
    out = {
        "header": {
            "gtfs_realtime_version": feed.header.gtfs_realtime_version,
            "incrementality": int(feed.header.incrementality) if feed.header.HasField("incrementality") else None,
            "timestamp": int(feed.header.timestamp) if feed.header.HasField("timestamp") else None,
        },
        "entity_count": len(feed.entity),
        "entities_preview": [],
    }

    for ent in feed.entity[:max_entities]:
        e: Dict[str, Any] = {"id": ent.id or None}
        if ent.is_deleted:
            e["is_deleted"] = True

        if ent.HasField("trip_update"):
            tu = ent.trip_update
            e["trip_update"] = {
                "trip": {
                    "trip_id": tu.trip.trip_id or None,
                    "route_id": tu.trip.route_id or None,
                    "direction_id": tu.trip.direction_id if tu.trip.HasField("direction_id") else None,
                    "start_time": tu.trip.start_time or None,
                    "start_date": tu.trip.start_date or None,
                },
                "vehicle": {"id": tu.vehicle.id or None, "label": tu.vehicle.label or None},
                "timestamp": int(tu.timestamp) if tu.HasField("timestamp") else None,
                "delay": int(tu.delay) if tu.HasField("delay") else None,
                "stop_time_updates": [],
            }
            # Bound stop updates too
            for stu in tu.stop_time_update[:50]:
                e["trip_update"]["stop_time_updates"].append({
                    "stop_id": stu.stop_id or None,
                    "arrival": {
                        "time": int(stu.arrival.time) if stu.arrival.HasField("time") else None,
                        "delay": int(stu.arrival.delay) if stu.arrival.HasField("delay") else None,
                    } if stu.HasField("arrival") else None,
                    "departure": {
                        "time": int(stu.departure.time) if stu.departure.HasField("time") else None,
                        "delay": int(stu.departure.delay) if stu.departure.HasField("delay") else None,
                    } if stu.HasField("departure") else None,
                })

        if ent.HasField("vehicle"):
            vp = ent.vehicle
            e["vehicle_position"] = {
                "trip": {
                    "trip_id": vp.trip.trip_id or None,
                    "route_id": vp.trip.route_id or None,
                } if vp.HasField("trip") else None,
                "vehicle": {"id": vp.vehicle.id or None, "label": vp.vehicle.label or None} if vp.HasField("vehicle") else None,
                "timestamp": int(vp.timestamp) if vp.HasField("timestamp") else None,
                "position": {
                    "latitude": vp.position.latitude if vp.HasField("position") else None,
                    "longitude": vp.position.longitude if vp.HasField("position") else None,
                    "bearing": vp.position.bearing if vp.HasField("position") and vp.position.HasField("bearing") else None,
                    "speed": vp.position.speed if vp.HasField("position") and vp.position.HasField("speed") else None,
                },
                "current_status": int(vp.current_status) if vp.HasField("current_status") else None,
                "stop_id": vp.stop_id or None,
            }

        if ent.HasField("alert"):
            al = ent.alert
            e["alert"] = {
                "cause": int(al.cause) if al.HasField("cause") else None,
                "effect": int(al.effect) if al.HasField("effect") else None,
                "header_text": al.header_text.translation[0].text if al.header_text.translation else None,
                "description_text": al.description_text.translation[0].text if al.description_text.translation else None,
            }

        out["entities_preview"].append(e)

    txt = json.dumps(out, ensure_ascii=False, indent=2)
    return "[GTFS-RT decoded preview]\n" + txt[:200_000]


I will also create a helper function for if the file is of type GTFS - because in that case I want to extract all the files inside of the zip file and add them all to the list of extracted files, separately.

In [198]:
import zipfile
from typing import List

def _extract_gtfs_zip_docs(
    path: str,
    *,
    max_rows: int,
    max_members: int = 250, # avoid pathological zips with thousands of files
) -> List[str]:
    """
    Extract GTFS zip into a list of docs (one per internal .txt/.csv file),
    reading only the first (1 header + max_rows data rows) lines per file.

    NOTE: In GTFS, each file is CSV-formatted text but usually named *.txt.
    """

    def _safe_member(name: str) -> bool:
        name = name.replace("\\", "/")
        if name.startswith("/") or name.startswith("../") or "/../" in name:
            return False
        return True

    def _decode_line(b: bytes) -> str:
        # Try common encodings per line; fallback to replacement
        for enc in ("utf-8-sig", "utf-8", "latin-1"):
            try:
                return b.decode(enc)
            except UnicodeDecodeError:
                continue
        return b.decode("utf-8", errors="replace")

    docs: List[str] = []

    with zipfile.ZipFile(path, "r") as zf:
        for info in zf.infolist()[:max_members]:
            # skip directories
            if getattr(info, "is_dir", lambda: info.filename.endswith("/"))():
                continue

            name = info.filename
            if not _safe_member(name):
                continue

            name_lower = name.lower()
            if not (name_lower.endswith(".txt") or name_lower.endswith(".csv")):
                continue

            # Stream first lines only (avoid reading whole member)
            lines: List[str] = []
            try:
                with zf.open(info, "r") as fp:
                    # Keep header + max_rows data rows => total lines = 1 + max_rows
                    # If max_rows is meant to include header, change to max_rows instead.
                    max_lines = max_rows + 1

                    for i, raw_line in enumerate(fp):
                        if i >= max_lines:
                            break
                        lines.append(_decode_line(raw_line).rstrip("\r\n"))

            except Exception:
                # If a specific member fails, skip it rather than failing the entire zip
                continue

            if not lines:
                continue

            text = "\n".join(lines).strip()
            docs.append(f"[GTFS file: {name} | preview_lines={len(lines)}]\n{text}")

    return docs


In [199]:
import os, re, json
import pandas as pd
from pypdf import PdfReader
import mimetypes
from urllib.parse import urlparse, unquote

_CD_FILENAME_RE = re.compile(r'filename\*?=(?:UTF-8\'\')?"?([^";]+)"?')

def _safe_filename(name: str) -> str:
    name = unquote(name)
    name = re.sub(r"[^a-zA-Z0-9._-]+", "_", name).strip("._")
    return name or "resource"

def _filename_from_content_disposition(cd: str) -> str | None:
    if not cd:
        return None
    m = _CD_FILENAME_RE.search(cd)
    return m.group(1).strip() if m else None

def _filename_from_url(url: str) -> str | None:
    path = urlparse(url).path
    if not path:
        return None
    base = os.path.basename(path.rstrip("/"))
    return base or None

def download_file(url: str, out_dir: str) -> str:
    os.makedirs(out_dir, exist_ok=True)

    with requests.get(url, stream=True, timeout=60, allow_redirects=True) as r:
        r.raise_for_status()

        # 1) Best: Content-Disposition filename
        cd = r.headers.get("Content-Disposition", "")
        name = _filename_from_content_disposition(cd)

        # 2) Next: final redirected URL filename
        if not name:
            name = _filename_from_url(r.url)

        # 3) Next: original URL filename
        if not name:
            name = _filename_from_url(url)

        name = _safe_filename(name or "resource")

        # If still no extension, infer from Content-Type
        root, ext = os.path.splitext(name)
        if not ext:
            ctype = (r.headers.get("Content-Type") or "").split(";")[0].strip().lower()
            guessed_ext = mimetypes.guess_extension(ctype)  # e.g. 'application/zip' -> '.zip'
            if guessed_ext:
                name = root + guessed_ext

        path = os.path.join(out_dir, name)

        # Avoid overwriting if same filename repeats
        if os.path.exists(path):
            base, ext2 = os.path.splitext(path)
            i = 2
            while os.path.exists(f"{base}__{i}{ext2}"):
                i += 1
            path = f"{base}__{i}{ext2}"

        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    return path

def extract_text_from_file(
    path: str,
    max_rows: int = 200,
    resource: Optional[Dict[str, Any]] = None,
) -> Union[str, List[str]]:
    lower = path.lower()

    # CSV
    if lower.endswith(".csv"):
        df = pd.read_csv(path, nrows=max_rows)
        return f"[CSV preview: first {len(df)} rows]\n" + df.to_csv(index=False)

    # JSON
    if lower.endswith(".json") or lower.endswith(".jsonl"):
        if lower.endswith(".jsonl"):
            # take first N lines
            lines = []
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                for i, line in enumerate(f):
                    if i >= max_rows:
                        break
                    lines.append(line.strip())
            return "[JSONL preview]\n" + "\n".join(lines)
        else:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                obj = json.load(f)
            txt = json.dumps(obj, ensure_ascii=False, indent=2)
            # keep it bounded
            return txt[:200_000]

    # TXT / MD
    if lower.endswith(".txt") or lower.endswith(".md"):
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()[:200_000]

    # PDF
    if lower.endswith(".pdf"):
        reader = PdfReader(path)
        pages = []
        for i, p in enumerate(reader.pages[:20]):  # bound pages
            pages.append(p.extract_text() or "")
        return "[PDF extracted text]\n" + "\n\n".join(pages)[:200_000]

    # ZIP (GTFS-aware)
    if lower.endswith(".zip"):
        r = resource or {}
        extras = r.get("extras") or {}

        fmt = str(r.get("format") or "")
        mime = str(r.get("mime") or "")
        desc = str(r.get("description") or "")
        analysis_mime = str(extras.get("analysis:mime-type") or "")

        combined = f"{fmt} {mime} {desc} {analysis_mime}".lower()
        is_gtfs = "gtfs" in combined

        if is_gtfs:
            docs = _extract_gtfs_zip_docs(path, max_rows=max_rows, max_members=250)
            if docs:
                return docs
            return f"[GTFS ZIP detected but no .txt/.csv extracted: {os.path.basename(path)}]"

        return f"[Unsupported binary format for text extraction: {os.path.basename(path)}]"

    if lower.endswith(".bin") or lower.endswith(".pb") or lower.endswith(".protobuf"):
        if _looks_like_gtfs_rt(resource):
            return extract_gtfs_rt_preview(path, max_entities=max_rows)
        return f"[Unsupported binary format for text extraction: {os.path.basename(path)}]"

    # Fallback: try decode as text
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()[:200_000]
    except Exception:
        return f"[Unsupported binary format for text extraction: {os.path.basename(path)}]"



So the above function (which checks the ending of each url path) is not actually entirely correct because it **assumes** that the download url will always contain the ending (e.g., .csv) at the end, but this is not true, as oftentimes the url **does not** end with the file extension but is still used to download that file

In [200]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def chunk_text(text: str, chunk_size: int = 2400, overlap: int = 200) -> List[str]:
    text = text.replace("\r", "\n")
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i+chunk_size])
        i += max(1, chunk_size - overlap)
    return chunks

class LocalTfidfRetriever:
    def __init__(self, documents: List[str]):
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.docs = documents
        self.mat = self.vectorizer.fit_transform(documents)

    def topk(self, query: str, k: int = 6) -> List[Tuple[int, float, str]]:
        q = self.vectorizer.transform([query])
        sims = cosine_similarity(q, self.mat)[0]
        idxs = sims.argsort()[::-1][:k]
        return [(int(i), float(sims[i]), self.docs[int(i)]) for i in idxs]


In [201]:
class GenerateQuestions(dspy.Signature):
    """Generate realistic user questions that AGORA could answer using this dataset and its resources."""
    dataset_metadata: str = dspy.InputField(desc="Metadata about the dataset: title, description, org, license, etc.")
    extracted_text_previews: str = dspy.InputField(desc="Short previews of extracted dataset resources, showing only the first N characters of each document.")
    # vvvv changed "top relevant" to "random"
    #retrieved_context: str = dspy.InputField(desc="Top relevant excerpts from the dataset resources (sample rows/text).")
    retrieved_context: str = dspy.InputField(desc="Random excerpts from the dataset resources (sample rows/text).")
    project_description: str = dspy.InputField(desc="AGORA project description & typical workflow constraints.")
    # #      had to modify this a little bit after noticing that the LLM returns a lot of questions which include
    # # VVVV specific data field names (eg. prix-max, prix-min, train-no, etc.), which is unrealistic
    # questions: str = dspy.OutputField(desc="A numbered list of 15-30 user questions. Each should be actionable, data-driven, and realistic. Remember that the user usually does not have any concrete information about the data, such as field names, or even the dataset as a whole. Only generate questiont that a user might ask which might PERTAIN to this dataset, assuming the user has no knowledge of the dataset's recoures or it's content, or even it's name or any information about it.")
    # ^^^^ DELETEED this because it just wasn't cutting it. I felt like the answers were still very technical. So I have this now:
    questions: str = dspy.OutputField(
    desc=(
        "A numbered list of 15–30 realistic end-user questions, "
        "Each question must be phrased in natural, non-technical language and reflect "
        "real decision-making or feasibility analysis (e.g., location suitability, demand, "
        "infrastructure, regulation, competition, or trends).\n\n"

        "The user has NO knowledge of:\n"
        "- the dataset’s name\n"
        "- available datasets\n"
        "- column names or schema\n"
        "- specific data fields or codes\n\n"

        "DO NOT mention dataset structures, variables, column names, or technical terms. "
        "Questions should sound like what a business owner, planner, or citizen would ask, "
        "expecting AGORA to find and analyze relevant government data automatically.\n\n"

        "Each question should plausibly be answerable using this dataset (possibly combined "
        "with others), but without assuming the user knows that this dataset exists."
      )
    )


class DatasetQuestionGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.cot = dspy.ChainOfThought(GenerateQuestions)

    def forward(self, dataset_metadata: str, extracted_text_previews: str, retrieved_context: str, project_description: str):
        return self.cot(
            dataset_metadata=dataset_metadata,
            extracted_text_previews=extracted_text_previews,
            retrieved_context=retrieved_context,
            project_description=project_description,
        )


In [202]:
# ---- Put your AGORA project description text here (from the PDF) ----
# Key points: agentic open-data reasoning, feasibility questions (zoning/demographics/competition/etc.), web platform workflow. :contentReference[oaicite:5]{index=5}
AGORA_PROJECT_DESCRIPTION = """
The Agent-based Government Open-data Reasoning Assistant (AGORA) project focuses on
building an intelligent web platform where AI agents autonomously analyze open government
data to answer complex user queries. Instead of manually searching through government
portals, users can simply ask natural language questions—such as “What’s the feasibility of
opening a restaurant in Algiers?”—and receive comprehensive, actionable, data-driven
reports.
"""

# ---- This function is deprecated - because it uses top-k chunks which are related to the retireival query.
# ---- This is unnecassary.

# def generate_questions_for_dataset(dataset_id: str,
#                                   download_dir: str = "/content/dg_resources",
#                                   max_files: int = 6,
#                                   top_k_chunks: int = 8) -> str:
#     # 1) Fetch dataset JSON + URLs
#     ds = get_dataset_payload(dataset_id)
#     urls = extract_resource_urls(ds)

#     # 2) Download + extract text (bounded)
#     texts = []
#     for url in urls[:max_files]:
#         try:
#             path = download_file(url, download_dir)
#             text = extract_text_from_file(path)
#             texts.append(f"=== FILE: {os.path.basename(path)} ===\nSOURCE_URL: {url}\n\n{text}")
#         except Exception as e:
#             texts.append(f"=== FILE DOWNLOAD FAILED ===\nURL: {url}\nERROR: {e}")

#     # 3) Build chunks + retriever
#     all_chunks = []
#     for t in texts:
#         all_chunks.extend(chunk_text(t))

#     retriever = LocalTfidfRetriever(all_chunks if all_chunks else ["(no extracted content)"])

#     # 4) Retrieve most relevant chunks for “question generation”
#     #    (Query focuses on producing user-facing questions that AGORA would answer.)
#     retrieval_query = (
#         "Generate realistic user questions that this dataset could help answer within AGORA. "
#         "Focus on feasibility/analysis queries (demographics, zoning, competition, infrastructure, trends)."
#     )
#     top = retriever.topk(retrieval_query, k=top_k_chunks)
#     retrieved_context = "\n\n".join([f"[chunk #{i} | score={s:.3f}]\n{c}" for i, s, c in top])

#     # 5) Dataset metadata summary
#     dataset_metadata = summarize_dataset_metadata(ds)

#     # 6) DSPy generation
#     program = DatasetQuestionGenerator()
#     pred = program(
#         dataset_metadata=dataset_metadata,
#         retrieved_context=retrieved_context,
#         project_description=AGORA_PROJECT_DESCRIPTION,
#     )

#     return pred.questions


In [203]:
#dataset_id = "66178e5e022893ce023d5f04"  # <-- fill in a real data.gouv.fr dataset id
#questions = generate_questions_for_dataset(dataset_id)
#print(questions)


I am adding additional features now that let me see the DSPy history as well as how many tokens were used. It also verifies file downloads.

In [204]:
import logging

# More verbose internal logs (optional but useful)
logging.getLogger("dspy").setLevel(logging.DEBUG)  # DSPy FAQ mentions this approach. :contentReference[oaicite:1]{index=1}

# Track token usage (DSPy 2.6.16+)
dspy.configure(track_usage=True)  # usage can be read from Prediction.get_lm_usage(). :contentReference[oaicite:2]{index=2}


In [205]:
import os, hashlib
from typing import List, Dict, Any

def sha1_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def verify_downloads(downloaded_paths: List[str]) -> List[Dict[str, Any]]:
    report = []
    for p in downloaded_paths:
        item = {"path": p, "exists": os.path.exists(p)}
        if item["exists"]:
            item["bytes"] = os.path.getsize(p)
            item["sha1"] = sha1_file(p)
        report.append(item)
    return report

def print_download_report(report: List[Dict[str, Any]]):
    print("\nDOWNLOAD REPORT")
    print("=" * 80)
    for r in report:
        if not r["exists"]:
            print(f"❌ Missing: {r['path']}")
        else:
            print(f"✅ {r['path']} | {r['bytes']} bytes | sha1={r['sha1']}")


In [206]:
from typing import List
def print_text_previews(extracted_texts: List[str], max_chars: int = 800):
    print("\nEXTRACTED TEXT PREVIEWS (this will get injected into the LLM call)")
    print("=" * 80)
    for i, t in enumerate(extracted_texts, start=1):
        preview = (t[:max_chars] + "…") if len(t) > max_chars else t
        print(
        f"\n--- Extracted doc #{i} ✅ --- "
        f"(total_extracted_chars_from_file={len(t)}, preview_chars={min(len(t), max_chars)}) ---\n"
        f"{preview}"
    )

def string_text_previews(extracted_texts: List[str], max_chars: int = 800) -> str:
    lines = []
    for i, t in enumerate(extracted_texts, start=1):
        preview = (t[:max_chars] + "…") if len(t) > max_chars else t
        lines.append(
            f"\n--- Extracted doc #{i} (preview_chars={len(preview)}) ---\n{preview}"
        )
    return "\n".join(lines)



In [207]:
import random

def generate_questions_for_dataset_debug(
    dataset_id: str,
    download_dir: str = "/content/dg_resources",
    #max_files: int = 6,  <-- I have removed this max_files variable,
    #                         otherwise it would go in this line:
    #                         for url in urls[:max_files]: ...
    top_k_chunks: int = 8,
    show_history_n: int = 1,
    show_previews: bool = True
):
    ds = get_dataset_payload(dataset_id)
    urls, urls_no = extract_resource_urls(ds)

    downloaded_paths = []
    extracted_texts = []

    # Download + extract
    for url in urls:
        try:
            path = download_file(url, download_dir)
            downloaded_paths.append(path)

            text = extract_text_from_file(path)

            # don't really know if I need the if-else statement here since
            # it's really only used when the url type is GTFS, which would
            # be zip, and it would probably be in the urls-no list instead
            # ... which is the code right under this for-loop ...
            if isinstance(text, list):
              extracted_texts.extend(text)   # GTFS ZIP: one per internal file
            else:
              extracted_texts.append(text)   # normal case

            # vvv REMOVED because I felt it was unecessary.
            #extracted_texts.append(
            #    f"=== FILE: {os.path.basename(path)} ===\nSOURCE_URL: {url}\n\n{text}"
            #)
        except Exception as e:
            extracted_texts.append(f"=== FILE FAILED ===\nURL: {url}\nERROR: {e}")

    for item in urls_no:
        url = item["url"]
        res = item["resource"]
        try:
            path = download_file(url, download_dir)   # same function
            downloaded_paths.append(path)

            text = extract_text_from_file(path, resource=res)
            if isinstance(text, list):
              extracted_texts.extend(text)   # GTFS ZIP: one per internal file
            else:
              extracted_texts.append(text)   # normal case

        except Exception as e:
            extracted_texts.append(f"=== FILE FAILED ===\nURL: {url}\nERROR: {e}")


    # Proof #1: downloaded files exist
    report = verify_downloads(downloaded_paths)
    print_download_report(report)

    # Proof #1.5: show some of what you extracted
    if show_previews:
        print_text_previews(extracted_texts, max_chars=800)

    # ADDED to show text previews (which usually includes headers)
    extracted_text_previews = string_text_previews(extracted_texts, max_chars=800)

    # vvvvv DELETED this retriever because I felt like it was unnecassary. New one is below it.
    # # Build retriever
    # all_chunks = []
    # for t in extracted_texts:
    #     all_chunks.extend(chunk_text(t))

    # retriever = LocalTfidfRetriever(all_chunks if all_chunks else ["(no extracted content)"])
    # retrieval_query = (
    #     "Generate realistic user questions that this dataset could help answer within AGORA. "
    #     "Focus on feasibility/analysis queries (demographics, zoning, competition, infrastructure, trends)."
    # )
    # top = retriever.topk(retrieval_query, k=top_k_chunks)
    # retrieved_context = "\n\n".join([f"[chunk #{i} | score={s:.3f}]\n{c}" for i, s, c in top])

    # Build chunks
    all_chunks = []
    for t in extracted_texts:
        all_chunks.extend(chunk_text(t))

    # Safety fallback
    if not all_chunks:
        all_chunks = ["(no extracted content)"]

    # Randomly sample chunks
    k = min(top_k_chunks, len(all_chunks))
    selected_chunks = random.sample(all_chunks, k)

    retrieved_context = "\n\n".join(
        f"[random chunk #{i}]\n{c}"
        for i, c in enumerate(selected_chunks)
    )

    print("\nRETRIEVED CONTEXT (this will get injected into the LLM call)")
    print("=" * 80)
    print(retrieved_context)

    dataset_metadata = summarize_dataset_metadata(ds)

    # Run DSPy program
    program = DatasetQuestionGenerator()
    pred = program(
        dataset_metadata=dataset_metadata,
        extracted_text_previews=extracted_text_previews,
        retrieved_context=retrieved_context,
        project_description=AGORA_PROJECT_DESCRIPTION,
    )

    # Proof #2: Inspect the exact prompt(s) DSPy sent to the LM
    # DSPy provides inspect_history utility for LLM invocations. :contentReference[oaicite:3]{index=3}
    if show_history_n and show_history_n > 0:
        print("\nDSPy LLM CALL HISTORY (recent)")
        print("=" * 80)
        dspy.inspect_history(n=show_history_n)

    # Optional: token usage (if supported by your DSPy version + enabled)
    try:
        usage = pred.get_lm_usage()
        print("\nLM USAGE")
        print("=" * 80)
        print(usage)
    except Exception:
        pass

    return {
        "questions": pred.questions,
        "downloaded_paths": downloaded_paths,
        "download_report": report,
        "extracted_texts": extracted_texts,
        "retrieved_context": retrieved_context,
        "dataset_metadata": dataset_metadata,
    }


The above code snippet is the MOST IMPORTANT in this entire notebook. This is where we are actually doing the whole process, everything up until this one is just helper functions.  Note that the returned things in this function include: questions, downloaded_paths, etc.

In the next code snippet we are testing out this function, which will also display the DSPy history - very useful for actually understanding what is going on

In [208]:
dataset_id = "5c65bad88b4c4143d45c9d5c"  # fill this
out = generate_questions_for_dataset_debug(dataset_id, show_history_n=1)



DOWNLOAD REPORT
✅ /content/dg_resources/palmbus-cannes-fr__5.zip | 38886406 bytes | sha1=a6175c1b4f29a886de942a83b8c4c2aabeb5146f
✅ /content/dg_resources/palmbus-cannes-gtfs-rt-service-alert__5.bin | 15 bytes | sha1=275bc59e0547e394591a0ccdd29ae95f90d28268
✅ /content/dg_resources/palmbus-cannes-gtfs-rt-vehicle-position__5.bin | 15 bytes | sha1=275bc59e0547e394591a0ccdd29ae95f90d28268
✅ /content/dg_resources/palmbus-cannes-gtfs-rt-trip-update__5.bin | 1013 bytes | sha1=ee232e64b9fa0009f4e9fda07c7cc1bb9b8ee496

EXTRACTED TEXT PREVIEWS (this will get injected into the LLM call)

--- Extracted doc #1 ✅ --- (total_extracted_chars_from_file=73, preview_chars=73) ---
[Unsupported binary format for text extraction: palmbus-cannes-fr__5.zip]

--- Extracted doc #2 ✅ --- (total_extracted_chars_from_file=183, preview_chars=183) ---
[GTFS-RT decoded preview]
{
  "header": {
    "gtfs_realtime_version": "2.0",
    "incrementality": 0,
    "timestamp": 1770150955
  },
  "entity_count": 0,
  "entitie

The next code snippet gives us additional information about our LLM calls using DSPy.

In [209]:
print("LM history length:", len(lm.history), "   --> (Note: this is probably number of times that the LLM was called IN TOTAL - not just in the last run)")
print("Last call keys:", lm.history[-1].keys())
print("Last call usage:", lm.history[-1].get("usage"))
print("Last call cost:", lm.history[-1].get("cost"))


LM history length: 1    --> (Note: this is probably number of times that the LLM was called IN TOTAL - not just in the last run)
Last call keys: dict_keys(['prompt', 'messages', 'kwargs', 'response', 'outputs', 'usage', 'cost', 'timestamp', 'uuid', 'model', 'response_model', 'model_type'])
Last call usage: {'completion_tokens': 1203, 'prompt_tokens': 4024, 'total_tokens': 5227, 'completion_tokens_details': CompletionTokensDetailsWrapper(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=320, rejected_prediction_tokens=0, text_tokens=None, image_tokens=None), 'prompt_tokens_details': PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None)}
Last call cost: 0.003412


This is just to copy dataset summary:

In [210]:
ds = get_dataset_payload(dataset_id)
dataset_metadata = summarize_dataset_metadata(ds)
print(dataset_metadata)

ID: 5c65bad88b4c4143d45c9d5c
Title: Horaires théoriques et temps réel  (GTFS & GTFS-RT) du réseau Palmbus - Cannes Pays de Lérins
Description_short: None
Description: Ce jeu de données présente les éléments descriptifs de l’offre de transport du réseau PALMBUS (lignes, arrêts, horaires,).
Ces données sont fournies au format GTFS et horaires temps-réel au format GTFS-RT.
Les Transports à la Demande (TAD) sont inclus.
Organization: Palmbus (slug=palmbus, id=5c65a43a634f41156b2de37e)
License: odc-odbl
Frequency: monthly
Created_at: 2019-02-14T20:00:40.136000+00:00
Last_update: 2026-02-03T19:55:52.022000+00:00
Tags: autobus, bus, busazur, cacpl, cannes, cannes-pays-de-lerins, donnees, gtfs, gtfs-real-time, gtfs-rt, le-cannet, lerins, mandelieu, mobilite, mougins, opendata, palmbus, temps-reel, theoule, transport-de-voyageurs, transports, transports-et-mobilite, urbain
URI: https://www.data.gouv.fr/api/1/datasets/horaires-theoriques-et-temps-reel-gtfs-gtfs-rt-du-reseau-palmbus-cannes-pays-d

This is just to copy the generated questions:

In [211]:
print(out["questions"])

1. Is public transport frequent enough near my proposed café location in central Cannes to attract steady lunchtime foot traffic on weekdays?
2. For a small hotel in Mandelieu, how convenient would it be for guests to reach Cannes train station or the airport by public transport in the evening?
3. How long would it take, on average during morning peak, for commuters to travel from Le Cannet to the main employment areas in Cannes using the bus network?
4. Are there parts of Mougins that have poor day‑time or evening bus coverage that might discourage customers from visiting a new retail store?
5. Which bus stops are consistently served on time, and which ones have frequent delays or cancellations that could affect punctual workers or time‑sensitive deliveries?
6. If we plan a weekend outdoor festival in Lérins, what is the public transport capacity and timetable coverage for transporting attendees from the city center and nearby towns?
7. Where are the best locations for a new park‑and‑